<a href="https://www.kaggle.com/code/ameythakur20/ai-mathematical-olympiad-environment-setup" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# AI Mathematical Olympiad: Environment Setup

This notebook configures the local environment for the AI Mathematical Olympiad (AIMO) competition. It establishes the necessary directory structures, downloads critical LLM-related dependencies for offline serving, and stages tiktoken encodings.

**Author:** [Amey Thakur](https://www.kaggle.com/ameythakur20)

**Objective:** Create a portable wheel-house and encoding artifact for high-performance mathematical reasoning models.

## 1. Environment Configuration

Primary directories for dependency management and encoding storage are initialized within the Kaggle working directory to ensure a structured staging area.

In [ ]:
import os
from pathlib import Path

# Workspace initialization for wheel-house and encodings
WHEEL_DIR = Path("/kaggle/working/wheels")
ENCODING_DIR = Path("/kaggle/working/tiktoken_encodings")

# Ensure persistence directories exist
WHEEL_DIR.mkdir(parents=True, exist_ok=True)
ENCODING_DIR.mkdir(parents=True, exist_ok=True)

print(f"[STATUS] Target wheel directory initialized at: {WHEEL_DIR}")
print(f"[STATUS] Encoding directory initialized at: {ENCODING_DIR}")

## 2. Dependency Resolution

Specific versions of Unsloth, TRL, and VLLM are harvested along with their binary trees. Tiktoken encodings are acquired from public endpoints to support local tokenizer initialization.

In [ ]:
# External dependency harvesting for offline inference pipelines
# We prioritize binary wheels for Python 3.12 to match the Kaggle environment
!pip download \
    --dest {WHEEL_DIR} \
    --python-version 3.12 \
    --only-binary=:all: \
    --extra-index-url https://download.pytorch.org/whl/cu128 \
    --extra-index-url https://pypi.nvidia.com \
    unsloth \
    trl \
    vllm==0.11.2 \
    openai_harmony

In [ ]:
# Optimization: Pruning redundant Ray artifacts to reduce archival overhead
!rm -rf /kaggle/working/ray*

In [ ]:
%%bash
# Harvesting tiktoken encodings for model-specific tokenization outside of internet-enabled sessions

wget -q -O /kaggle/working/tiktoken_encodings/o200k_base.tiktoken \
    'https://openaipublic.blob.core.windows.net/encodings/o200k_base.tiktoken'

wget -q -O /kaggle/working/tiktoken_encodings/cl100k_base.tiktoken \
    'https://openaipublic.blob.core.windows.net/encodings/cl100k_base.tiktoken'

echo "[STATUS] Tiktoken encodings successfully staged."

In [ ]:
import pandas as pd
from glob import glob

# Professional verification layer to ensure artifact integrity
def verify_artifacts(directory, pattern="*"):
    files = glob(os.path.join(directory, pattern))
    data = []
    for f in files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        data.append({"Artifact": os.path.basename(f), "Size (MB)": f"{size_mb:.2f}"})
    return pd.DataFrame(data)

print("\n--- WHEEL ARTIFACT SUMMARY ---")
print(verify_artifacts(WHEEL_DIR).head(10))
print(f"\nTotal wheels harvested: {len(glob(os.path.join(WHEEL_DIR, '*')))}")

print("\n--- ENCODING ARTIFACT SUMMARY ---")
print(verify_artifacts(ENCODING_DIR))

In [ ]:
# Aggregating dependencies into a single compressed transport archive
!tar -czf /kaggle/working/wheels.tar.gz -C /kaggle/working wheels tiktoken_encodings

# Cleanup: Purging staging directories to maintain workspace hygiene
!rm -rf /kaggle/working/wheels
!rm -rf /kaggle/working/tiktoken_encodings

print(f"[SUCCESS] Artifacts consolidated into: /kaggle/working/wheels.tar.gz ({os.path.getsize('/kaggle/working/wheels.tar.gz') / (1024*1024):.2f} MB)")

## 3. Analysis Summary

The environment setup is complete. All specified wheels and encodings have been aggregated into a compressed archive (`wheels.tar.gz`), enabling offline installation for the AIMO competition pipeline.